# Pipeline de Predicción – Proyecto Final MLOps17

| Proyecto | Rol | Recursos |
|---|---|---|
| `mlops17-fuente` | Fuente | BigQuery `hist_table.input_data` |
| `mlops17-507600` | Procesamiento | Vertex Pipeline, bucket `mlops17_vertex02` (modelo), dataset `tmp_procesamiento` (TABLE_OUTPUT) |
| `mlops17-produccion` | Producción | BigQuery `modelo_propension.ba_modelo_propension_inscripcion` |

Flujo: **Validate Data → Componente 1 (BigQuery SQL → TABLE_OUTPUT) → Componente 2 (Python: modelo GCS → predicción → BigQuery producción)**

In [ ]:
# Ejecutar una sola vez y luego reiniciar el entorno de ejecución
!pip install -q "kfp>=2.7,<3" google-cloud-pipeline-components google-cloud-aiplatform google-cloud-storage

In [ ]:
from typing import NamedTuple

from kfp import dsl, compiler
from kfp.dsl import component
from google.cloud import aiplatform
from google.cloud import storage
from google_cloud_pipeline_components.v1.vertex_notification_email import VertexNotificationEmailOp

## Configuración

In [ ]:
PROJECT_PROCESAMIENTO = "mlops17-507600"
REGION = "us-central1"
BUCKET = "mlops17_vertex02"
PIPELINE_ROOT = f"gs://{BUCKET}/proyecto_final/pipeline_root"
PIPELINE_BLOB = "proyecto_final/pipeline_prediction.json"
SERVICE_ACCOUNT = f"mlops17-vertex-process@{PROJECT_PROCESAMIENTO}.iam.gserviceaccount.com"
EMAIL = "<tu_correo@dominio.com>"  # correo que recibe la notificación del pipeline

PARAMS = {
    "project": PROJECT_PROCESAMIENTO,
    "source_table": "mlops17-fuente.hist_table.input_data",
    "features_table": f"{PROJECT_PROCESAMIENTO}.tmp_procesamiento.selected_features",
    "table_output": f"{PROJECT_PROCESAMIENTO}.tmp_procesamiento.table_output",
    "prediction_table": "mlops17-produccion.modelo_propension.ba_modelo_propension_inscripcion",
    "path_model": f"gs://{BUCKET}/proyecto_final/model/linear_regression.joblib",
}

## Componentes

In [ ]:
@component
def error_op(msg: str):
    raise ValueError(msg)

### Validate Data
Valida, con la cuenta de servicio del pipeline, que **existan** los recursos y que **se tenga acceso**:
- tabla fuente (proyecto fuente): existe, tiene filas y se puede leer → *BigQuery Data Viewer*
- tabla de features: existe y se puede leer
- dataset de producción: existe y es accesible → *BigQuery Data Editor*
- modelo en Cloud Storage: existe y se puede leer → *Storage Object Viewer*

In [ ]:
@component(packages_to_install=["google-cloud-bigquery", "google-cloud-storage"])
def validate_data(
    project: str,
    source_table: str,
    features_table: str,
    prediction_table: str,
    path_model: str,
) -> NamedTuple("Outputs", [("condition", str)]):
    from google.cloud import bigquery, storage

    client = bigquery.Client(project=project)
    errores = []

    def validar_lectura(table_id, requiere_filas=False):
        try:
            table = client.get_table(table_id)          # ¿existe?
            if requiere_filas and table.num_rows == 0:
                errores.append(f"{table_id} está vacía")
                return
            client.query(f"SELECT 1 FROM `{table_id}` LIMIT 1").result()  # ¿se puede leer?
            print(f"OK lectura: {table_id} ({table.num_rows} filas)")
        except Exception as e:
            errores.append(f"Sin acceso o no existe {table_id}: {e}")

    # 1. Tabla fuente y tabla de features
    validar_lectura(source_table, requiere_filas=True)
    validar_lectura(features_table, requiere_filas=True)

    # 2. Dataset de producción (la tabla puede no existir en la primera ejecución)
    dataset_id = prediction_table.rsplit(".", 1)[0]
    try:
        client.get_dataset(dataset_id)
        print(f"OK dataset producción: {dataset_id}")
    except Exception as e:
        errores.append(f"Sin acceso o no existe el dataset {dataset_id}: {e}")

    # 3. Modelo en Cloud Storage
    try:
        bucket_name, blob_name = path_model.replace("gs://", "").split("/", 1)
        blob = storage.Client(project=project).bucket(bucket_name).blob(blob_name)
        if not blob.exists():
            errores.append(f"No existe el modelo {path_model}")
        else:
            print(f"OK modelo: {path_model}")
    except Exception as e:
        errores.append(f"Sin acceso al modelo {path_model}: {e}")

    for err in errores:
        print("ERROR:", err)

    condition = "false" if errores else "true"
    return (condition,)

### Componente 1: Extracción y procesamiento (BigQuery SQL)
Lee la tabla del proyecto fuente, aplica las transformaciones y genera **TABLE_OUTPUT** con las variables del modelo.

In [ ]:
@component(packages_to_install=["google-cloud-bigquery"])
def process_data(project: str, source_table: str, features_table: str, table_output: str):
    from google.cloud import bigquery

    client = bigquery.Client(project=project)

    features = [
        row.string_field_0
        for row in client.query(f"SELECT string_field_0 FROM `{features_table}`").result()
    ]
    # Transformaciones: selección de variables, nulos a 0 y casteo a FLOAT64
    columnas = ",\n      ".join(
        f"CAST(IFNULL(`{f}`, 0) AS FLOAT64) AS `{f}`" for f in features
    )

    sql = f"""
    CREATE OR REPLACE TABLE `{table_output}` AS
    SELECT
      ROW_NUMBER() OVER () AS id_registro,
      {columnas}
    FROM `{source_table}`
    """
    print(sql)
    client.query(sql).result()
    print(f"TABLE_OUTPUT generada: {client.get_table(table_output).num_rows} filas")

### Componente 2: Predicción del modelo (Python)
Lee TABLE_OUTPUT, carga el modelo desde Cloud Storage, predice y guarda el histórico en BigQuery del proyecto de producción.

In [ ]:
@component(
    packages_to_install=[
        "google-cloud-bigquery",
        "google-cloud-storage",
        "pandas",
        "scikit-learn",
        "joblib",
        "db-dtypes",
        "pyarrow",
        "pytz",
    ],
)
def prediction(project: str, table_output: str, prediction_table: str, path_model: str):
    from datetime import datetime
    from io import BytesIO

    import pandas as pd
    from google.cloud import bigquery, storage
    from joblib import load
    from pytz import timezone

    TZ = timezone("America/Lima")
    client = bigquery.Client(project=project)

    # 1. Leer los datos procesados
    df = client.query(f"SELECT * FROM `{table_output}` ORDER BY id_registro").to_dataframe()

    # 2. Cargar el modelo desde Cloud Storage
    def load_model_from_gcs(path_model):
        bucket_name, blob_name = path_model.replace("gs://", "").split("/", 1)
        blob = storage.Client(project=project).bucket(bucket_name).blob(blob_name)
        return load(BytesIO(blob.download_as_bytes()))

    model = load_model_from_gcs(path_model)

    # 3. Ejecutar la predicción (mismo orden de columnas que en el entrenamiento)
    features = list(getattr(model, "feature_names_in_", df.columns.drop("id_registro")))
    preds = model.predict(df[features])

    # 4. Guardar resultados en BigQuery del proyecto de producción (histórico -> append)
    user_id = client.query("SELECT SESSION_USER()").to_dataframe().iloc[0, 0]
    now = datetime.now(TZ)
    result = pd.DataFrame({
        "id_registro": df["id_registro"].astype("int64"),
        "prediction": preds.astype("float64"),
        "creation_user": user_id,
        "process_date": now.date(),
        "load_date": now.replace(tzinfo=None),
    })

    job_config = bigquery.LoadJobConfig(
        write_disposition="WRITE_APPEND",
        schema=[
            bigquery.SchemaField("id_registro", "INT64"),
            bigquery.SchemaField("prediction", "FLOAT64"),
            bigquery.SchemaField("creation_user", "STRING"),
            bigquery.SchemaField("process_date", "DATE"),
            bigquery.SchemaField("load_date", "DATETIME"),
        ],
    )
    client.load_table_from_dataframe(result, prediction_table, job_config=job_config).result()
    print(f"{len(result)} predicciones guardadas en {prediction_table}")

## Pipeline

In [ ]:
@dsl.pipeline(
    name="pipeline-prediction-propension",
    description="BigQuery fuente -> TABLE_OUTPUT -> modelo GCS -> BigQuery producción",
    pipeline_root=PIPELINE_ROOT,
)
def main_pipeline(
    project: str,
    source_table: str,
    features_table: str,
    table_output: str,
    prediction_table: str,
    path_model: str,
):
    notify_email_task = VertexNotificationEmailOp(recipients=[EMAIL])
    notify_email_task.set_display_name("Notification Email")

    with dsl.ExitHandler(notify_email_task, name="Execute pipeline prediction"):

        validate_tables_job = validate_data(
            project=project,
            source_table=source_table,
            features_table=features_table,
            prediction_table=prediction_table,
            path_model=path_model,
        )
        validate_tables_job.set_display_name("Validate Data")

        with dsl.If(validate_tables_job.outputs["condition"] == "false", name="no-execute"):
            error_op(msg="No se logró validar las tablas o los accesos. Revisar el log de Validate Data.")

        with dsl.If(validate_tables_job.outputs["condition"] == "true", name="execute"):

            process_task = process_data(
                project=project,
                source_table=source_table,
                features_table=features_table,
                table_output=table_output,
            )
            process_task.set_display_name("COMPONENTE_1_BIGQUERY_PROCESS")

            prediction_task = prediction(
                project=project,
                table_output=table_output,
                prediction_table=prediction_table,
                path_model=path_model,
            ).after(process_task)
            prediction_task.set_display_name("COMPONENTE_2_PREDICTION_MODEL")

## Compilar y subir a Cloud Storage

In [ ]:
compiler.Compiler().compile(
    pipeline_func=main_pipeline,
    package_path="pipeline_prediction.json"
)

In [ ]:
def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    storage_client = storage.Client(project=PROJECT_PROCESAMIENTO)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"Archivo {source_file_name} subido a gs://{bucket_name}/{destination_blob_name}")

upload_to_gcs(BUCKET, "pipeline_prediction.json", PIPELINE_BLOB)

## Ejecutar con la cuenta de servicio personalizada

In [ ]:
aiplatform.init(project=PROJECT_PROCESAMIENTO, location=REGION)

job = aiplatform.PipelineJob(
    display_name="pipeline-prediction-propension",
    template_path=f"gs://{BUCKET}/{PIPELINE_BLOB}",
    enable_caching=False,
    project=PROJECT_PROCESAMIENTO,
    location=REGION,
    parameter_values=PARAMS,
)

print("submit pipeline job ...")
job.submit(service_account=SERVICE_ACCOUNT)